In [76]:
pip install mne

In [77]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mne
import seaborn as sns






In [78]:
##Importa dataset da Kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yosftag/open-nuro-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'open-nuro-dataset' dataset.
Path to dataset files: /kaggle/input/open-nuro-dataset


In [79]:
#Stampo file.tsv con partecipanti
import os
tsv_path = "/root/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/participants.tsv"
print(tsv_path)
participants = pd.read_csv(tsv_path, sep='\t')
print(participants)




/root/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/participants.tsv
   participant_id Gender  Age Group  MMSE
0         sub-001      F   57     A    16
1         sub-002      F   78     A    22
2         sub-003      M   70     A    14
3         sub-004      F   67     A    20
4         sub-005      M   70     A    22
..            ...    ...  ...   ...   ...
83        sub-084      F   71     F    24
84        sub-085      M   64     F    26
85        sub-086      M   49     F    26
86        sub-087      M   73     F    24
87        sub-088      M   55     F    24

[88 rows x 5 columns]


In [80]:
#Informazioni su EEG
file_path = '/root/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set'
raw = mne.io.read_raw_eeglab(file_path, preload=True)
print(raw)
print(raw.info)
print(raw.ch_names)

<RawEEGLAB | sub-001_task-eyesclosed_eeg.set, 19 x 299900 (599.8 s), ~43.5 MiB, data loaded>
<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T3, T4, T5, ...
 chs: 19 EEG
 custom_ref_applied: False
 dig: 22 items (3 Cardinal, 19 EEG)
 highpass: 0.0 Hz
 lowpass: 250.0 Hz
 meas_date: unspecified
 nchan: 19
 projs: []
 sfreq: 500.0 Hz
>
['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']


In [ ]:
#Divido in epoche di 30 s con 50% overlap e calcolo PSD, poi divido in bande
from pathlib import Path

bands = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0), #bande spettrali EEG
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

rows = []

files = sorted(Path("/root/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/derivatives/").rglob("*.set"))

for file in files:

    raw = mne.io.read_raw_eeglab(
        file,
        preload=True,
        verbose=False   #carico file EEG.set
    )

    epoch = mne.make_fixed_length_epochs(
        raw,
        duration=30,    #divido in epoche 30s overlap 15s
        overlap=15,
        preload=True
    )

    psd = epoch.compute_psd(
        method="welch",
        fmin=0.5,         #calcolo Power spectral density
        fmax=45
    )

    psd_db = 10 * np.log10(psd.get_data())  #converto in dB

    freqs = psd.freqs

    features = {
        "subject": file.parent.parent.name
    }
        #creo file con le feature canale_banda
    for ch_idx, ch_name in enumerate(epoch.ch_names):

        for band_name, (fmin, fmax) in bands.items():

            idx = (freqs >= fmin) & (freqs <= fmax) #scelgo solo la frequenza della banda

            features[f"{ch_name}_{band_name}"] = (
                psd_db[:, ch_idx, :][:, idx].mean()
            )

    rows.append(features)

df_features = pd.DataFrame(rows)

Not setting metadata
38 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 38 events and 15000 original time points ...


In [ ]:
#Unisco al df con le feature i partecipanti del file tsv
df_final = df_features.merge(participants,left_on="subject",right_on="participant_id",how="left")


In [ ]:
#Assegno un valore numerico ai gruppi per semplicità
df_final['Group'] = df_final['Group'].map({'A': 1, 'F': 2, 'C': 0})



In [ ]:
df_final.head()

In [ ]:
#Elimino colonne senza feauture e con caratteri testuali

columns = df_final.drop(columns=['subject', 'participant_id', 'Group','Age','Gender','MMSE'])
##Calcolo media per ogni colonna
for col in columns:
    df_final[f"{col}_mean"] = df_final[col].mean()
#Creo media tra bande psd
theta_columns = [col for col in df_final.columns if col.endswith('_theta')]
alpha_columns = [col for col in df_final.columns if col.endswith('_alpha')]
beta_columns = [col for col in df_final.columns if col.endswith('_beta')]
gamma_columns = [col for col in df_final.columns if col.endswith('_gamma')]
delta_columns = [col for col in df_final.columns if col.endswith('_delta')]
theta = (np.abs(df_final[theta_columns])**2).mean(axis=1)
alpha = (np.abs(df_final[alpha_columns])**2).mean(axis=1)
beta = (np.abs(df_final[beta_columns])**2).mean(axis=1)
gamma = (np.abs(df_final[gamma_columns])**2).mean(axis=1)
delta = (np.abs(df_final[delta_columns])**2).mean(axis=1)
#Calcolor rapporti tra bande spettrali
dt = delta/theta  #delta-theta band power ratio
db = delta/beta   #delta-beta band power ratio
da = delta/alpha  #delta-alpha band power ratio
ta = theta/alpha  #theta-alpha band power ratio
tb = theta/beta   #theta-beta band power ratio
ab = alpha/beta   #alpha-beta band power ratio



#aggiungo le nuove colonne di feature al file df_final
df_final['DTBPR'] = dt
df_final['DBBPR'] = db
df_final['DABPR'] = da
df_final['TABPR'] = ta
df_final['TBBPR'] = tb
df_final['ABBPR'] = ab
df_final.head()


Cerco delle feature che possono essere maggiormente correlate rispetto ad altre


In [ ]:

corr = df_final.drop(
columns=['subject', 'participant_id', 'Group','Age','Gender','MMSE']  #Elimino le colonne che non
                                                                #contengono feauture
).corrwith(df_final['Group']) #Calcolo correlazione con gruppi HC, Alzhaimer,Fronto-temporal Dementia

corr = corr.abs().sort_values(ascending=False)
top_corr = corr.head(60)  #Considero le prime 50

top_corr.plot(kind='bar')
plt.title('Correlation with Group')
plt.xlabel('Feature')
plt.ylabel('Correlation')
plt.show()

In [ ]:
#Creo un dataframe solo con le feature di interesse e il gruppo associato
selected_feature = top_corr.index
df_selected = df_final[list(selected_feature)+ ['Group']+['participant_id']]
df_selected.head()
df_selected.shape

In [ ]:
#Creo dataframe con le features per i miei modelli ML e uno con i target
X_full_features = df_selected.drop(columns=['Group','participant_id'])
y_full = df_selected['Group']
print("dimensioni di X_full_features:", X_full_features.shape)
print("dimensioni di y_full :",y_full.shape)

In [ ]:
#Divido il dataset in 70% per training e 30% per test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_full_features, y_full,
    test_size=0.3,
    random_state=42,
    stratify=y_full
)
print("Train shape X:", X_train.shape, " y:", y_train.shape)
print("Test shape  X:", X_test.shape,  " y:", y_test.shape)



In [ ]:
def squared_inverse_distance(distances):
    return 1.0 / (distances**2 + 1e-12)

In [ ]:
def gausssian (distances, sigma = 1.0):
    return np.exp(-distances**2)/(2*sigma**2)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

#Definisco Pipeline per standardizzare e scegliere il modello
pipe_knn = Pipeline(
    [('scaler', StandardScaler()),
     ('knn', KNeighborsClassifier())]
)
#Scelgo i parametri da ottimizzare
param_knn = {
    'knn__n_neighbors': [2,4,6],
    'knn__weights': ['distance', squared_inverse_distance,gausssian ],
    'knn__metric': ['cosine','euclidean'],
    'knn__algorithm': ['brute']
}
#Eseguo cross-validation per evitare data leakage
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
#Uso GridSearch per ottimizzare gli iperparametri del modello
grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_knn,
    scoring="roc_auc_ovr",
    cv=sgkf,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
grid_knn.fit(X_train, y_train,groups=df_selected.loc[X_train.index, 'participant_id'])#Aggiungo partecipant_id per evitare che lo stesso partecipante
                                                                                      #sia in training e test
print(
    "Migliori parametri k-NN:",
    grid_knn.best_params_,
)
#Uso il modello con iperparametri ottimizzati per fae previsioni
y_pred_knn = grid_knn.predict(X_test)
y_proba_knn = grid_knn.predict_proba(X_test)
#Calcolo ROC-AUC
roc_auc_knn = roc_auc_score(
    y_test, y_proba_knn, multi_class='ovr', average='macro'
)
print("ROC AUC (test) con k-NN:", roc_auc_knn)
class_names = ['Control', 'Alzheimer', 'FTD']
#Genero confusion matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title('Confusion Matrix for k-NN Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print(" metriche k-NN Model:")
print(classification_report(y_test, y_pred_knn, target_names=class_names))


In [ ]:
from sklearn.svm import SVC

pipe_svc = Pipeline(
    [('scaler', StandardScaler()), ('svc', SVC(probability=True))]
)

param_svc = {
    'svc__kernel': ['poly','rbf'],
    'svc__C': [0.01,0.1,1,10,100],
    'svc__gamma':['scale','auto'],
    'svc__class_weight':['balanced',None],
    'svc__degree':[2,3,4] #funzione di grado 2 per kernel polinomiale
}
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)

grid_svc = GridSearchCV(
    estimator=pipe_svc,
    param_grid=param_svc,
    scoring="roc_auc_ovr",
    cv=sgkf,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

grid_svc.fit(X_train, y_train,groups=df_selected.loc[X_train.index, 'participant_id'])
print("Migliori parametri SVC", grid_svc.best_params_)

y_pred_svc = grid_svc.predict(X_test)
y_proba_svc = grid_svc.predict_proba(X_test)
roc_auc_svc = roc_auc_score(
    y_test, y_proba_svc, multi_class='ovr', average='macro'
)
print("ROC AUC (test) con SVC:", roc_auc_svc)

cm_svc = confusion_matrix(y_test, y_pred_svc)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_svc,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title('Confusion Matrix for SVC Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
#Stampo Accuracy e f1 score del modello SVC
print(" metriche SVC Model:")
print(classification_report(y_test, y_pred_svc, target_names=class_names))






In [ ]:
from sklearn.metrics import RocCurveDisplay
#STAMPO curve ROC per i modelli
n_classes = len(class_names)
fig,axes = plt.subplots(1,n_classes,figsize=(15,10))

for i in range(n_classes):
    #PLOT ROC curve per k-NN
    RocCurveDisplay.from_predictions(
        y_test==i,
        y_proba_knn[:,i],
        name=f'k-NN (Class {class_names[i]})',
        ax=axes[i]
    )
    #PLOT ROC curve per SVC
    RocCurveDisplay.from_predictions(
        y_test==i,
        y_proba_svc[:,i],
        name=f'SVC (Class {class_names[i]})',
        ax=axes[i]
    )
    axes[i].plot([0,1],[0,1],'k--',label='Chance (AUC = 0.5)')
    axes[i].set_title(f'ROC Curve for Class: {class_names[i]}')
    axes[i].legend(loc='lower right')

plt.tight_layout()
plt.show()